# Continuation pack (play-a-hand) — GPU on Kaggle

Re-solves the **continuation** content (linked flop→turn→river hands; villain plays its
solved main line) and writes the signed `continuation_seed` pack.

Runs on the **batched GPU solver** (`--solver gpu` → `BatchedGPUCFR`, CuPy). The trajectory
extraction was ported onto it and an independent oracle proved it correct at streets=3 —
where the old `MultiStreetSpike` had a river-betting bug. Faster **and** more correct.

Uses **float32** on GPU (float64 is ~1/32 speed on a T4). Pick a **GPU** session; **Internet On**.

In [ ]:
# Fail fast BEFORE cloning/solving if this isn't a working GPU session — so a misconfigured
# runtime can't silently burn CPU time on the NumPy fallback.
import cupy, subprocess
ndev = cupy.cuda.runtime.getDeviceCount()
assert ndev > 0, 'No CUDA device — set the runtime Accelerator to GPU.'
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],
                     capture_output=True, text=True).stdout)
print(f'CuPy sees {ndev} GPU(s) — good to go')

In [ ]:
# Clone the solver source (needs Internet On).
!rm -rf /kaggle/working/poker && git clone -q --depth 1 https://github.com/tian-chaiyaporn2/poker_offline_trainer /kaggle/working/poker
import sys; sys.path.insert(0, '/kaggle/working/poker/src')
import subprocess
print('source ready @', subprocess.run(['git','-C','/kaggle/working/poker','rev-parse','--short','HEAD'],
                                       capture_output=True, text=True).stdout.strip())

In [ ]:
# Solve the continuation library on GPU (float32) and write the signed pack.
#   FLOPS  : curated (flop,turn,river) runouts. Max 4 today; add tuples to CURATED in
#            demo/gen_continuation.py (~line 45) then raise FLOPS.
#   N      : combos sampled per range.  ITERS: CFR iterations (run prints stable per flop -> all True).
#   The run prints `solver backend=cupy dtype=float32` — if it says numpy, the GPU isn't engaged.
import subprocess, os
FLOPS, N, ITERS, VERSION = 4, 80, 600, 'continuation_seed'
env = {**os.environ, 'PYTHONPATH': 'src'}
subprocess.run(
    ['python', 'demo/gen_continuation.py', '--solver', 'gpu', '--dtype', 'float32',
     '--flops', str(FLOPS), '--n', str(N), '--iters', str(ITERS), '--version', VERSION],
    cwd='/kaggle/working/poker', env=env, check=True)

In [ ]:
# Expose the three pack files for download (Output panel).
import shutil, os
VERSION = 'continuation_seed'
base = '/kaggle/working/poker/output/packs'
files = [f'flop_pack_{VERSION}.db', f'flop_pack_{VERSION}.db.gz', f'build_report_{VERSION}.json']
if not os.path.exists(os.path.join(base, files[0])):
    raise SystemExit('No pack written -- check the solve cell for stable=False / errors.')
for f in files:
    shutil.copy(os.path.join(base, f), os.path.join('/kaggle/working', f))
print('DOWNLOAD from /kaggle/working:')
for f in files:
    print('  %-42s %d KB' % (f, os.path.getsize(os.path.join('/kaggle/working', f)) // 1024))